In [1]:
# Cell 1: Imports, device, constants

import copy
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import transforms
from torchvision.datasets import ImageFolder

# Device + shared constants used throughout training
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES = 10
INPUT_SHAPE = (3, 180, 180)
BATCH_SIZE = 32
SEED = 42
LEARNING_RATE_FC = 1e-3
LEARNING_RATE_CNN = 1e-4
LEARNING_RATE_RMSPROP = 1e-4
DEBUG_EPOCHS = 1

print("Using device:", DEVICE)


Using device: cpu


In [2]:
import torch
from torch.utils.data import DataLoader, random_split
from torchvision import transforms
from torchvision.datasets import ImageFolder

# Paths
IMAGE_DIR = "Data/images_original"

# Hyperparameters
BATCH_SIZE = 32
SEED = 42

# Image preprocessing required by the coursework
image_transform = transforms.Compose([
    transforms.Resize((180, 180)),
    transforms.ToTensor()
])

# Load spectrogram image dataset
image_dataset = ImageFolder(
    root=IMAGE_DIR,
    transform=image_transform
)

# Check classes
print("Classes:", image_dataset.classes)
print("Class to index:", image_dataset.class_to_idx)
print("Total image samples:", len(image_dataset))

# Train / validation / test split: 70% / 20% / 10%
total_size = len(image_dataset)
train_size = int(0.7 * total_size)
val_size = int(0.2 * total_size)
test_size = total_size - train_size - val_size

train_dataset, val_dataset, test_dataset = random_split(
    image_dataset,
    [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(SEED)
)

print("Train size:", len(train_dataset))
print("Validation size:", len(val_dataset))
print("Test size:", len(test_dataset))

# DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

# Check one batch
images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Example labels:", labels[:10])


Classes: ['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock']
Class to index: {'blues': 0, 'classical': 1, 'country': 2, 'disco': 3, 'hiphop': 4, 'jazz': 5, 'metal': 6, 'pop': 7, 'reggae': 8, 'rock': 9}
Total image samples: 999
Train size: 699
Validation size: 199
Test size: 101
Image batch shape: torch.Size([32, 3, 180, 180])
Label batch shape: torch.Size([32])
Example labels: tensor([7, 1, 6, 6, 6, 6, 0, 9, 4, 6])


In [3]:
# Cell 3: Utility functions

def set_seed(seed=SEED):
    """Set random seeds for reproducibility across Python, NumPy, and PyTorch."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def count_parameters(model):
    """Return number of trainable parameters in a model."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def smoke_test_model(model, input_shape=INPUT_SHAPE, batch_size=4, num_classes=NUM_CLASSES, device=DEVICE):
    """
    Quick shape/device sanity check for a model.
    Ensures output shape is [batch_size, num_classes].
    """
    model = model.to(device)
    model.eval()
    with torch.no_grad():
        x = torch.randn(batch_size, *input_shape).to(device)
        y = model(x)

    assert y.shape == (batch_size, num_classes), (
        f"Smoke test failed: expected {(batch_size, num_classes)}, got {tuple(y.shape)}"
    )
    print(f"Smoke test passed. Output shape: {tuple(y.shape)}")
    print(f"Trainable parameters: {count_parameters(model):,}")


In [4]:
# Cell 4: Training and evaluation functions

criterion = nn.CrossEntropyLoss()
results_records = []


def train_one_epoch(model, loader, optimizer, criterion, device=DEVICE):
    """Train for one epoch and return average training loss."""
    model.train()
    running_loss = 0.0
    total = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        total += images.size(0)

    return running_loss / total


def evaluate(model, loader, criterion, device=DEVICE):
    """Evaluate model and return (avg_loss, accuracy_percent)."""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return running_loss / total, 100.0 * correct / total


def train_model(model, model_name, architecture, optimizer_name, epochs, train_loader, val_loader, criterion, lr=LEARNING_RATE_FC, device=DEVICE):
    """
    Full training loop with validation tracking.
    Returns model loaded with best validation checkpoint and history dictionary.
    """
    set_seed(SEED)
    model = model.to(device)

    if optimizer_name.lower() == "rmsprop":
        optimizer = optim.RMSprop(model.parameters(), lr=lr)
    elif optimizer_name.lower() == "adam":
        optimizer = optim.Adam(model.parameters(), lr=lr)
    else:
        raise ValueError(f"Unsupported optimizer: {optimizer_name}")

    history = {"train_loss": [], "val_loss": [], "val_acc": []}
    best_val_acc = -1.0
    best_state = copy.deepcopy(model.state_dict())

    for epoch in range(1, epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
        val_loss, val_acc = evaluate(model, val_loader, criterion, device)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = copy.deepcopy(model.state_dict())

        print(
            f"[{model_name}] Epoch {epoch}/{epochs} | "
            f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%"
        )

    model.load_state_dict(best_state)
    return model, history


def test_model(model, loader, criterion, device=DEVICE):
    """Evaluate trained model on test set."""
    test_loss, test_acc = evaluate(model, loader, criterion, device)
    return test_loss, test_acc


def append_result(model_name, architecture, optimizer_name, epochs, history, test_loss, test_acc):
    """Append one run result to shared records list."""
    best_epoch_idx = int(np.argmax(history["val_acc"]))
    results_records.append({
        "model_name": model_name,
        "architecture": architecture,
        "optimizer": optimizer_name,
        "epochs": epochs,
        "final_training_loss": history["train_loss"][-1],
        "final_validation_loss": history["val_loss"][-1],
        "final_validation_accuracy": history["val_acc"][-1],
        "best_validation_accuracy": max(history["val_acc"]),
        "best_epoch": best_epoch_idx + 1,
        "test_loss": test_loss,
        "test_accuracy": test_acc,
    })


In [5]:
# Cell 5: Net1 model definition only

class Net1(nn.Module):
    """Net1: fully connected network with exactly two hidden layers."""
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        # Net1 has many parameters because a 180x180 RGB image is flattened directly.
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(3 * 180 * 180, 512)
        self.fc2 = nn.Linear(512, 128)
        self.fc3 = nn.Linear(128, num_classes)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.flatten(x)
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)
        return x


In [8]:
# Cell 6: Net1 smoke test and optional 1-epoch debug run

set_seed(SEED)
net1 = Net1()
smoke_test_model(net1)

RUN_DEBUG_NET1 = True  # Set True to run 1 epoch quick debug training
if RUN_DEBUG_NET1:
    set_seed(SEED)
    net1_debug = Net1()
    net1_debug, net1_debug_history = train_model(
        model=net1_debug,
        model_name="Net1",
        architecture="FC(Flatten->512->128->10)",
        optimizer_name="Adam",
        epochs=DEBUG_EPOCHS,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        lr=LEARNING_RATE_FC,
    )
    dbg_test_loss, dbg_test_acc = test_model(net1_debug, test_loader, criterion)
    print(f"Net1 debug test loss: {dbg_test_loss:.4f}, test acc: {dbg_test_acc:.2f}%")
else:
    print("Net1 debug run skipped. Set RUN_DEBUG_NET1=True to execute.")


Smoke test passed. Output shape: (4, 10)
Trainable parameters: 49,833,866
[Net1] Epoch 1/1 | Train Loss: 7.6485 | Val Loss: 2.8138 | Val Acc: 19.10%
Net1 debug test loss: 2.9220, test acc: 12.87%


In [7]:
# Cell 7: Net1 50-epoch and 100-epoch training calls

RUN_FULL_NET1 = False  # Set True when you want full Net1 training
if RUN_FULL_NET1:
    for n_epochs in [50, 100]:
        set_seed(SEED)
        net1_run = Net1()
        net1_run, net1_history = train_model(
            model=net1_run,
            model_name="Net1",
            architecture="FC(Flatten->512->128->10)",
            optimizer_name="Adam",
            epochs=n_epochs,
            train_loader=train_loader,
            val_loader=val_loader,
            criterion=criterion,
            lr=LEARNING_RATE_FC,
        )
        net1_test_loss, net1_test_acc = test_model(net1_run, test_loader, criterion)
        append_result("Net1", "FC(Flatten->512->128->10)", "Adam", n_epochs, net1_history, net1_test_loss, net1_test_acc)
        print(f"Net1 ({n_epochs} epochs) test acc: {net1_test_acc:.2f}%")
else:
    print("Net1 full training skipped. Set RUN_FULL_NET1=True to execute.")


Net1 full training skipped. Set RUN_FULL_NET1=True to execute.


In [9]:
# Cell 8: Net2 model definition only

class Net2(nn.Module):
    """
    Net2: Coursework Figure 1 style CNN
    Input -> Conv+ReLU -> Conv+ReLU -> MaxPool
          -> Conv+ReLU -> Conv+ReLU -> MaxPool
          -> FC+ReLU -> FC output
    """
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

        with torch.no_grad():
            dummy = torch.zeros(1, *INPUT_SHAPE)
            flattened_dim = self.features(dummy).view(1, -1).size(1)

        self.classifier = nn.Sequential(
            nn.Linear(flattened_dim, 256),
            nn.ReLU(),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x


In [11]:
# Cell 9: Net2 smoke test and optional 1-epoch debug run

set_seed(SEED)
net2 = Net2()
smoke_test_model(net2)

RUN_DEBUG_NET2 = True
if RUN_DEBUG_NET2:
    set_seed(SEED)
    net2_debug = Net2()
    net2_debug, net2_debug_history = train_model(
        model=net2_debug,
        model_name="Net2",
        architecture="CNN(Fig1 4conv+2pool+fc256)",
        optimizer_name="Adam",
        epochs=DEBUG_EPOCHS,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        # Use lower LR for CNNs due to observed instability with larger LR.
        lr=LEARNING_RATE_CNN,
    )
    dbg_test_loss, dbg_test_acc = test_model(net2_debug, test_loader, criterion)
    print(f"Net2 debug test loss: {dbg_test_loss:.4f}, test acc: {dbg_test_acc:.2f}%")
else:
    print("Net2 debug run skipped. Set RUN_DEBUG_NET2=True to execute.")


Smoke test passed. Output shape: (4, 10)
Trainable parameters: 33,245,994
[Net2] Epoch 1/1 | Train Loss: 2.3552 | Val Loss: 2.2996 | Val Acc: 15.58%
Net2 debug test loss: 2.3089, test acc: 5.94%


In [12]:
# Cell 10: Net2 50-epoch and 100-epoch training calls

RUN_FULL_NET2 = False
if RUN_FULL_NET2:
    for n_epochs in [50, 100]:
        set_seed(SEED)
        net2_run = Net2()
        net2_run, net2_history = train_model(
            model=net2_run,
            model_name="Net2",
            architecture="CNN(Fig1 4conv+2pool+fc256)",
            optimizer_name="Adam",
            epochs=n_epochs,
            train_loader=train_loader,
            val_loader=val_loader,
            criterion=criterion,
            lr=LEARNING_RATE_CNN,
        )
        net2_test_loss, net2_test_acc = test_model(net2_run, test_loader, criterion)
        append_result("Net2", "CNN(Fig1 4conv+2pool+fc256)", "Adam", n_epochs, net2_history, net2_test_loss, net2_test_acc)
        print(f"Net2 ({n_epochs} epochs) test acc: {net2_test_acc:.2f}%")
else:
    print("Net2 full training skipped. Set RUN_FULL_NET2=True to execute.")


Net2 full training skipped. Set RUN_FULL_NET2=True to execute.


In [13]:
# Cell 11: Net3 model definition only

class Net3(nn.Module):
    """
    Net3: Same architecture as Net2, but with BatchNorm2d
    after each convolution and before ReLU.
    """
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

        with torch.no_grad():
            dummy = torch.zeros(1, *INPUT_SHAPE)
            flattened_dim = self.features(dummy).view(1, -1).size(1)

        self.classifier = nn.Sequential(
            nn.Linear(flattened_dim, 256),
            nn.ReLU(),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x


In [14]:
# Cell 12: Net3 smoke test and optional 1-epoch debug run

set_seed(SEED)
net3 = Net3()
smoke_test_model(net3)

RUN_DEBUG_NET3 = True
if RUN_DEBUG_NET3:
    set_seed(SEED)
    net3_debug = Net3()
    net3_debug, net3_debug_history = train_model(
        model=net3_debug,
        model_name="Net3",
        architecture="CNN+BN(Fig1 4conv+2pool+fc256)",
        optimizer_name="Adam",
        epochs=DEBUG_EPOCHS,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        # Use lower LR for CNN+BatchNorm due to unstable early debug losses.
        lr=LEARNING_RATE_CNN,
    )
    dbg_test_loss, dbg_test_acc = test_model(net3_debug, test_loader, criterion)
    print(f"Net3 debug test loss: {dbg_test_loss:.4f}, test acc: {dbg_test_acc:.2f}%")
else:
    print("Net3 debug run skipped. Set RUN_DEBUG_NET3=True to execute.")


Smoke test passed. Output shape: (4, 10)
Trainable parameters: 33,246,378
[Net3] Epoch 1/1 | Train Loss: 24.2430 | Val Loss: 3.7669 | Val Acc: 9.55%
Net3 debug test loss: 4.7888, test acc: 13.86%


In [15]:
# Cell 13: Net3 50-epoch and 100-epoch training calls

RUN_FULL_NET3 = False
if RUN_FULL_NET3:
    for n_epochs in [50, 100]:
        set_seed(SEED)
        net3_run = Net3()
        net3_run, net3_history = train_model(
            model=net3_run,
            model_name="Net3",
            architecture="CNN+BN(Fig1 4conv+2pool+fc256)",
            optimizer_name="Adam",
            epochs=n_epochs,
            train_loader=train_loader,
            val_loader=val_loader,
            criterion=criterion,
            lr=LEARNING_RATE_CNN,
        )
        net3_test_loss, net3_test_acc = test_model(net3_run, test_loader, criterion)
        append_result("Net3", "CNN+BN(Fig1 4conv+2pool+fc256)", "Adam", n_epochs, net3_history, net3_test_loss, net3_test_acc)
        print(f"Net3 ({n_epochs} epochs) test acc: {net3_test_acc:.2f}%")
else:
    print("Net3 full training skipped. Set RUN_FULL_NET3=True to execute.")


Net3 full training skipped. Set RUN_FULL_NET3=True to execute.


In [16]:
# Cell 14: Net4 setup using Net3 architecture and RMSProp

# Net4 must use the SAME architecture as Net3.
# We reuse Net3 directly to keep comparison fair.
# RMSprop is used only to satisfy the coursework requirement of same architecture + different optimizer.
Net4 = Net3
NET4_ARCH = "CNN+BN(Fig1 4conv+2pool+fc256)"
NET4_OPTIMIZER = "RMSprop"

print("Net4 setup complete: using Net3 architecture with RMSprop optimizer.")


Net4 setup complete: using Net3 architecture with RMSprop optimizer.


In [17]:
# Cell 15: Net4 smoke test and optional 1-epoch debug run

set_seed(SEED)
net4 = Net4()
smoke_test_model(net4)

RUN_DEBUG_NET4 = True
if RUN_DEBUG_NET4:
    set_seed(SEED)
    net4_debug = Net4()
    net4_debug, net4_debug_history = train_model(
        model=net4_debug,
        model_name="Net4",
        architecture=NET4_ARCH,
        optimizer_name=NET4_OPTIMIZER,
        epochs=DEBUG_EPOCHS,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        # Use lower LR for RMSprop because larger LR showed unstable early debug losses.
        lr=LEARNING_RATE_RMSPROP,
    )
    dbg_test_loss, dbg_test_acc = test_model(net4_debug, test_loader, criterion)
    print(f"Net4 debug test loss: {dbg_test_loss:.4f}, test acc: {dbg_test_acc:.2f}%")
else:
    print("Net4 debug run skipped. Set RUN_DEBUG_NET4=True to execute.")


Smoke test passed. Output shape: (4, 10)
Trainable parameters: 33,246,378
[Net4] Epoch 1/1 | Train Loss: 154.6020 | Val Loss: 4.9564 | Val Acc: 26.13%
Net4 debug test loss: 5.3167, test acc: 20.79%


In [18]:
# Cell 16: Net4 50-epoch and 100-epoch training calls

RUN_FULL_NET4 = False
if RUN_FULL_NET4:
    for n_epochs in [50, 100]:
        set_seed(SEED)
        net4_run = Net4()
        net4_run, net4_history = train_model(
            model=net4_run,
            model_name="Net4",
            architecture=NET4_ARCH,
            optimizer_name=NET4_OPTIMIZER,
            epochs=n_epochs,
            train_loader=train_loader,
            val_loader=val_loader,
            criterion=criterion,
            lr=LEARNING_RATE_RMSPROP,
        )
        net4_test_loss, net4_test_acc = test_model(net4_run, test_loader, criterion)
        append_result("Net4", NET4_ARCH, NET4_OPTIMIZER, n_epochs, net4_history, net4_test_loss, net4_test_acc)
        print(f"Net4 ({n_epochs} epochs) test acc: {net4_test_acc:.2f}%")
else:
    print("Net4 full training skipped. Set RUN_FULL_NET4=True to execute.")


Net4 full training skipped. Set RUN_FULL_NET4=True to execute.


In [19]:
# Cell 17: Results table and CSV saving

# Build DataFrame from completed training runs.
results_df = pd.DataFrame(results_records)

if results_df.empty:
    print("No full training runs have been logged yet.")
    print("Run Net1-Net4 full training cells with RUN_FULL_* = True, then rerun this cell.")
else:
    results_df = results_df.sort_values(["model_name", "epochs"]).reset_index(drop=True)
    display(results_df)

# Save CSV even if empty, so coursework file path is always created.
results_df.to_csv("results_image_models.csv", index=False)
print("Saved results to results_image_models.csv")


No full training runs have been logged yet.
Run Net1-Net4 full training cells with RUN_FULL_* = True, then rerun this cell.
Saved results to results_image_models.csv


In [ ]:
# Cell 18: Audio imports and constants
import os
import glob
import warnings
import copy

import librosa
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset, ConcatDataset, random_split

AUDIO_DIR = "Data/genres_original"
SAMPLE_RATE = 22050
DURATION = 30
N_MFCC = 40
MAX_AUDIO_TIME_STEPS = 1300
AUDIO_BATCH_SIZE = 16
AUDIO_DEBUG_EPOCHS = 1
AUDIO_FULL_EPOCHS = 20
GAN_DEBUG_EPOCHS = 1
GAN_FULL_EPOCHS = 20
AUDIO_LEARNING_RATE = 1e-3
GAN_LEARNING_RATE = 1e-4
NOISE_DIM = 100


In [ ]:
# Cell 19: AudioMFCCDataset class
class AudioMFCCDataset(Dataset):
    """Dataset that loads audio files and returns fixed-length MFCC sequences."""

    class_names = ["blues", "classical", "country", "disco", "hiphop", "jazz", "metal", "pop", "reggae", "rock"]

    def __init__(self, audio_dir=AUDIO_DIR):
        self.audio_dir = audio_dir
        self.class_to_idx = {name: idx for idx, name in enumerate(self.class_names)}
        self.samples = []
        self.labels = []

        for class_name in self.class_names:
            class_dir = os.path.join(self.audio_dir, class_name)
            wav_paths = sorted(glob.glob(os.path.join(class_dir, "*.wav")))
            for wav_path in wav_paths:
                self.samples.append(wav_path)
                self.labels.append(self.class_to_idx[class_name])

    def __len__(self):
        return len(self.samples)

    def _extract_mfcc(self, path):
        y, sr = librosa.load(path, sr=SAMPLE_RATE, duration=DURATION)
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC)
        mfcc = mfcc.T  # [time, n_mfcc]

        if mfcc.shape[0] < MAX_AUDIO_TIME_STEPS:
            pad_len = MAX_AUDIO_TIME_STEPS - mfcc.shape[0]
            mfcc = np.pad(mfcc, ((0, pad_len), (0, 0)), mode="constant")
        else:
            mfcc = mfcc[:MAX_AUDIO_TIME_STEPS, :]

        return torch.tensor(mfcc, dtype=torch.float32)

    def __getitem__(self, idx):
        path = self.samples[idx]
        label = self.labels[idx]

        try:
            feature = self._extract_mfcc(path)
        except Exception as e:
            warnings.warn(f"Failed to read/process audio file '{path}': {e}")
            feature = torch.zeros((MAX_AUDIO_TIME_STEPS, N_MFCC), dtype=torch.float32)

        return feature, torch.tensor(label, dtype=torch.long)


In [ ]:
# Cell 20: Audio dataset loading and PyTorch split
audio_dataset = AudioMFCCDataset(AUDIO_DIR)
print("Audio classes:", audio_dataset.class_names)
print("Class to index mapping:", audio_dataset.class_to_idx)
print("Total audio samples:", len(audio_dataset))

n_total = len(audio_dataset)
train_size = int(0.7 * n_total)
val_size = int(0.2 * n_total)
test_size = n_total - train_size - val_size

split_generator = torch.Generator().manual_seed(SEED)
train_audio_dataset, val_audio_dataset, test_audio_dataset = random_split(
    audio_dataset,
    [train_size, val_size, test_size],
    generator=split_generator,
)

train_audio_loader = DataLoader(train_audio_dataset, batch_size=AUDIO_BATCH_SIZE, shuffle=True)
val_audio_loader = DataLoader(val_audio_dataset, batch_size=AUDIO_BATCH_SIZE, shuffle=False)
test_audio_loader = DataLoader(test_audio_dataset, batch_size=AUDIO_BATCH_SIZE, shuffle=False)

print(f"Train samples: {len(train_audio_dataset)}")
print(f"Validation samples: {len(val_audio_dataset)}")
print(f"Test samples: {len(test_audio_dataset)}")

example_features, example_labels = next(iter(train_audio_loader))
print("One audio feature batch shape:", example_features.shape)
print("One label batch shape:", example_labels.shape)


In [ ]:
# Cell 21: Net5LSTM model definition
class Net5LSTM(nn.Module):
    def __init__(self, input_size=N_MFCC, hidden_size=128, num_layers=2, num_classes=10, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout,
        )
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        _, (h_n, _) = self.lstm(x)
        final_hidden = h_n[-1]
        logits = self.classifier(final_hidden)
        return logits


In [ ]:
# Cell 22: Audio training and evaluation functions
criterion_audio = nn.CrossEntropyLoss()
audio_results_records = []


def train_one_epoch_audio(model, loader, optimizer):
    model.train()
    running_loss = 0.0
    total = 0

    for features, labels in loader:
        features = features.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(features)
        loss = criterion_audio(outputs, labels)
        loss.backward()
        optimizer.step()

        batch_size = features.size(0)
        running_loss += loss.item() * batch_size
        total += batch_size

    return running_loss / max(total, 1)


def evaluate_audio(model, loader):
    model.eval()
    running_loss = 0.0
    total = 0
    correct = 0

    with torch.no_grad():
        for features, labels in loader:
            features = features.to(DEVICE)
            labels = labels.to(DEVICE)
            outputs = model(features)
            loss = criterion_audio(outputs, labels)

            batch_size = features.size(0)
            running_loss += loss.item() * batch_size
            total += batch_size
            correct += (outputs.argmax(dim=1) == labels).sum().item()

    avg_loss = running_loss / max(total, 1)
    accuracy = correct / max(total, 1)
    return avg_loss, accuracy


def train_audio_model(model, train_loader, val_loader, epochs, learning_rate=AUDIO_LEARNING_RATE):
    model = model.to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    history = {"train_loss": [], "val_loss": []}
    best_val_acc = -1.0
    best_epoch = -1
    best_state = None

    for epoch in range(1, epochs + 1):
        train_loss = train_one_epoch_audio(model, train_loader, optimizer)
        val_loss, val_acc = evaluate_audio(model, val_loader)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())

        print(
            f"Epoch {epoch}/{epochs} | Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}"
        )

    if best_state is not None:
        model.load_state_dict(best_state)

    final_val_loss, final_val_acc = evaluate_audio(model, val_loader)

    return model, {
        "train_loss": history["train_loss"][-1] if history["train_loss"] else None,
        "val_loss": history["val_loss"][-1] if history["val_loss"] else None,
        "final_validation_accuracy": final_val_acc,
        "best_validation_accuracy": best_val_acc,
        "best_epoch": best_epoch,
    }


def test_audio_model(model, test_loader):
    test_loss, test_acc = evaluate_audio(model, test_loader)
    return {"test_loss": test_loss, "test_accuracy": test_acc}


def append_audio_result(model_name, epochs, run_type, train_summary, test_summary):
    audio_results_records.append({
        "model": model_name,
        "epochs": epochs,
        "run_type": run_type,
        "train_loss": train_summary["train_loss"],
        "val_loss": train_summary["val_loss"],
        "final_validation_accuracy": train_summary["final_validation_accuracy"],
        "best_validation_accuracy": train_summary["best_validation_accuracy"],
        "best_epoch": train_summary["best_epoch"],
        "test_loss": test_summary["test_loss"],
        "test_accuracy": test_summary["test_accuracy"],
    })


In [ ]:
# Cell 23: Net5 smoke test and 1-epoch debug run
# Net5 uses MFCC sequences extracted from original audio samples.
# MFCC is used because raw waveform LSTM training is computationally expensive.

def smoke_test_audio_model(model, loader):
    model = model.to(DEVICE)
    model.eval()
    with torch.no_grad():
        features, _ = next(iter(loader))
        features = features.to(DEVICE)
        outputs = model(features)
    assert outputs.shape == (features.shape[0], 10), f"Unexpected output shape: {outputs.shape}"
    print("Smoke test passed. Output shape:", outputs.shape)


RUN_DEBUG_NET5 = True
RUN_FULL_NET5 = False

net5_debug_model = Net5LSTM()
smoke_test_audio_model(net5_debug_model, train_audio_loader)

if RUN_DEBUG_NET5:
    net5_debug_model = Net5LSTM()
    net5_debug_model, net5_debug_train_summary = train_audio_model(
        net5_debug_model,
        train_audio_loader,
        val_audio_loader,
        epochs=AUDIO_DEBUG_EPOCHS,
        learning_rate=AUDIO_LEARNING_RATE,
    )
    net5_debug_test_summary = test_audio_model(net5_debug_model, test_audio_loader)
    print(
        f"Net5 Debug Test -> Loss: {net5_debug_test_summary['test_loss']:.4f}, "
        f"Accuracy: {net5_debug_test_summary['test_accuracy']:.4f}"
    )
    append_audio_result("Net5LSTM", 1, "debug", net5_debug_train_summary, net5_debug_test_summary)


In [ ]:
# Cell 24: Net5 full training cell
RUN_FULL_NET5 = False

if RUN_FULL_NET5:
    net5_full_model = Net5LSTM()
    net5_full_model, net5_full_train_summary = train_audio_model(
        net5_full_model,
        train_audio_loader,
        val_audio_loader,
        epochs=AUDIO_FULL_EPOCHS,
        learning_rate=AUDIO_LEARNING_RATE,
    )
    net5_full_test_summary = test_audio_model(net5_full_model, test_audio_loader)
    print(
        f"Net5 Full Test -> Loss: {net5_full_test_summary['test_loss']:.4f}, "
        f"Accuracy: {net5_full_test_summary['test_accuracy']:.4f}"
    )
    append_audio_result("Net5LSTM", AUDIO_FULL_EPOCHS, "full", net5_full_train_summary, net5_full_test_summary)


In [ ]:
# Cell 25: Feature-level GAN dataset preparation
GAN_FEATURE_DIM = MAX_AUDIO_TIME_STEPS * N_MFCC

real_train_features = []
for batch_features, _ in train_audio_loader:
    real_train_features.append(batch_features)

real_train_features = torch.cat(real_train_features, dim=0)
real_train_features_flat = real_train_features.view(real_train_features.size(0), -1)

gan_train_loader = DataLoader(
    TensorDataset(real_train_features_flat),
    batch_size=AUDIO_BATCH_SIZE,
    shuffle=True,
)

print("Real training MFCC shape:", real_train_features.shape)
print("Flattened GAN feature dimension:", GAN_FEATURE_DIM)
print("Flattened tensor shape:", real_train_features_flat.shape)


In [ ]:
# Cell 26: Feature-level GAN models
class MFCCGenerator(nn.Module):
    def __init__(self, noise_dim=NOISE_DIM, output_dim=GAN_FEATURE_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(noise_dim, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, 512),
            nn.ReLU(inplace=True),
            nn.Linear(512, output_dim),
        )

    def forward(self, z):
        return self.net(z)


class MFCCDiscriminator(nn.Module):
    def __init__(self, input_dim=GAN_FEATURE_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(256, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.net(x)


In [ ]:
# Cell 27: GAN training function
def train_gan(generator, discriminator, gan_loader, epochs):
    generator = generator.to(DEVICE)
    discriminator = discriminator.to(DEVICE)

    bce = nn.BCELoss()
    g_optimizer = optim.Adam(generator.parameters(), lr=GAN_LEARNING_RATE)
    d_optimizer = optim.Adam(discriminator.parameters(), lr=GAN_LEARNING_RATE)

    for epoch in range(1, epochs + 1):
        g_epoch_loss = 0.0
        d_epoch_loss = 0.0
        total_batches = 0

        for (real_batch,) in gan_loader:
            real_batch = real_batch.to(DEVICE)
            batch_size = real_batch.size(0)

            real_targets = torch.ones((batch_size, 1), device=DEVICE)
            fake_targets = torch.zeros((batch_size, 1), device=DEVICE)

            # Train discriminator
            d_optimizer.zero_grad()
            real_pred = discriminator(real_batch)
            d_real_loss = bce(real_pred, real_targets)

            noise = torch.randn(batch_size, NOISE_DIM, device=DEVICE)
            fake_batch = generator(noise).detach()
            fake_pred = discriminator(fake_batch)
            d_fake_loss = bce(fake_pred, fake_targets)

            d_loss = d_real_loss + d_fake_loss
            d_loss.backward()
            d_optimizer.step()

            # Train generator
            g_optimizer.zero_grad()
            noise = torch.randn(batch_size, NOISE_DIM, device=DEVICE)
            generated = generator(noise)
            pred_generated = discriminator(generated)
            g_loss = bce(pred_generated, real_targets)
            g_loss.backward()
            g_optimizer.step()

            g_epoch_loss += g_loss.item()
            d_epoch_loss += d_loss.item()
            total_batches += 1

        print(
            f"GAN Epoch {epoch}/{epochs} | "
            f"D Loss: {d_epoch_loss / max(total_batches, 1):.4f} | "
            f"G Loss: {g_epoch_loss / max(total_batches, 1):.4f}"
        )

    return generator, discriminator


In [ ]:
# Cell 28: GAN debug run
RUN_DEBUG_GAN = True
RUN_FULL_GAN = False

generator = MFCCGenerator()
discriminator = MFCCDiscriminator()

if RUN_DEBUG_GAN:
    generator, discriminator = train_gan(generator, discriminator, gan_train_loader, epochs=GAN_DEBUG_EPOCHS)
    with torch.no_grad():
        noise = torch.randn(AUDIO_BATCH_SIZE, NOISE_DIM, device=DEVICE)
        generated_flat = generator(noise)
        generated_seq = generated_flat.view(-1, MAX_AUDIO_TIME_STEPS, N_MFCC)
    print("Generated debug tensor shape:", generated_seq.shape)


In [ ]:
# Cell 29: Create augmented training dataset for Net6
def generate_synthetic_mfcc(generator, num_samples):
    generator.eval()
    synth_batches = []
    generated_count = 0

    with torch.no_grad():
        while generated_count < num_samples:
            current_batch = min(AUDIO_BATCH_SIZE, num_samples - generated_count)
            noise = torch.randn(current_batch, NOISE_DIM, device=DEVICE)
            synth_flat = generator(noise)
            synth_seq = synth_flat.view(-1, MAX_AUDIO_TIME_STEPS, N_MFCC)
            synth_batches.append(synth_seq.cpu())
            generated_count += current_batch

    return torch.cat(synth_batches, dim=0)


train_indices = train_audio_dataset.indices
train_labels_tensor = torch.tensor([audio_dataset.labels[i] for i in train_indices], dtype=torch.long)

run_full_net6_flag = globals().get("RUN_FULL_NET6", False)
if run_full_net6_flag:
    num_synthetic_samples = len(train_audio_dataset)
else:
    num_synthetic_samples = min(256, len(train_audio_dataset))

synthetic_features = generate_synthetic_mfcc(generator, num_synthetic_samples)
rand_idx = torch.randint(low=0, high=len(train_labels_tensor), size=(num_synthetic_samples,))
synthetic_labels = train_labels_tensor[rand_idx]

real_features = real_train_features
real_labels = train_labels_tensor

aug_features = torch.cat([real_features, synthetic_features], dim=0)
aug_labels = torch.cat([real_labels, synthetic_labels], dim=0)

augmented_train_audio_dataset = TensorDataset(aug_features, aug_labels)
augmented_train_audio_loader = DataLoader(
    augmented_train_audio_dataset,
    batch_size=AUDIO_BATCH_SIZE,
    shuffle=True,
)

print("Number of real training samples:", len(real_features))
print("Number of synthetic samples:", len(synthetic_features))
print("Number of augmented training samples:", len(augmented_train_audio_dataset))


In [ ]:
# Cell 30: Net6 setup
# Net6 uses the same LSTM architecture as Net5.
# Net6 differs from Net5 only by GAN-augmented training data.
# Net6 augments the training set using GAN-generated MFCC-like features.
# This is a feature-level GAN approximation rather than raw waveform generation.
# Validation and test sets must remain real audio only.

def build_net6_model():
    return Net5LSTM()


In [ ]:
# Cell 31: Net6 smoke test and 1-epoch debug run
RUN_DEBUG_NET6 = True
RUN_FULL_NET6 = False

net6_debug_model = build_net6_model()
smoke_test_audio_model(net6_debug_model, augmented_train_audio_loader)

if RUN_DEBUG_NET6:
    net6_debug_model = build_net6_model()
    net6_debug_model, net6_debug_train_summary = train_audio_model(
        net6_debug_model,
        augmented_train_audio_loader,
        val_audio_loader,
        epochs=AUDIO_DEBUG_EPOCHS,
        learning_rate=AUDIO_LEARNING_RATE,
    )
    net6_debug_test_summary = test_audio_model(net6_debug_model, test_audio_loader)
    print(
        f"Net6 Debug Test -> Loss: {net6_debug_test_summary['test_loss']:.4f}, "
        f"Accuracy: {net6_debug_test_summary['test_accuracy']:.4f}"
    )
    append_audio_result("Net6LSTM_GANAug", 1, "debug", net6_debug_train_summary, net6_debug_test_summary)


In [ ]:
# Cell 32: Net6 full training cell
RUN_FULL_NET6 = False

if RUN_FULL_NET6:
    net6_full_model = build_net6_model()
    net6_full_model, net6_full_train_summary = train_audio_model(
        net6_full_model,
        augmented_train_audio_loader,
        val_audio_loader,
        epochs=AUDIO_FULL_EPOCHS,
        learning_rate=AUDIO_LEARNING_RATE,
    )
    net6_full_test_summary = test_audio_model(net6_full_model, test_audio_loader)
    print(
        f"Net6 Full Test -> Loss: {net6_full_test_summary['test_loss']:.4f}, "
        f"Accuracy: {net6_full_test_summary['test_accuracy']:.4f}"
    )
    append_audio_result("Net6LSTM_GANAug", AUDIO_FULL_EPOCHS, "full", net6_full_train_summary, net6_full_test_summary)


In [ ]:
# Cell 33: Audio results table and CSV saving
audio_results_df = pd.DataFrame(audio_results_records)
display(audio_results_df)
audio_results_df.to_csv("results_audio_models.csv", index=False)
print("Saved audio results to results_audio_models.csv")


In [ ]:
# Cell 34: Overnight run control cell
RUN_OVERNIGHT_NET5_NET6 = False

if RUN_OVERNIGHT_NET5_NET6:
    print("Starting overnight pipeline for Net5 and Net6...")

    # 1) Train Net5 full
    net5_overnight_model = Net5LSTM()
    net5_overnight_model, net5_overnight_train_summary = train_audio_model(
        net5_overnight_model,
        train_audio_loader,
        val_audio_loader,
        epochs=AUDIO_FULL_EPOCHS,
        learning_rate=AUDIO_LEARNING_RATE,
    )
    net5_overnight_test_summary = test_audio_model(net5_overnight_model, test_audio_loader)
    append_audio_result("Net5LSTM", AUDIO_FULL_EPOCHS, "overnight", net5_overnight_train_summary, net5_overnight_test_summary)

    # 2) Train GAN full
    overnight_generator = MFCCGenerator()
    overnight_discriminator = MFCCDiscriminator()
    overnight_generator, overnight_discriminator = train_gan(
        overnight_generator,
        overnight_discriminator,
        gan_train_loader,
        epochs=GAN_FULL_EPOCHS,
    )

    # 3) Generate synthetic MFCC features
    overnight_synth_features = generate_synthetic_mfcc(overnight_generator, len(train_audio_dataset))
    overnight_rand_idx = torch.randint(low=0, high=len(train_labels_tensor), size=(len(train_audio_dataset),))
    overnight_synth_labels = train_labels_tensor[overnight_rand_idx]

    # 4) Train Net6 full on augmented dataset
    overnight_aug_features = torch.cat([real_train_features, overnight_synth_features], dim=0)
    overnight_aug_labels = torch.cat([train_labels_tensor, overnight_synth_labels], dim=0)
    overnight_aug_dataset = TensorDataset(overnight_aug_features, overnight_aug_labels)
    overnight_aug_loader = DataLoader(overnight_aug_dataset, batch_size=AUDIO_BATCH_SIZE, shuffle=True)

    net6_overnight_model = build_net6_model()
    net6_overnight_model, net6_overnight_train_summary = train_audio_model(
        net6_overnight_model,
        overnight_aug_loader,
        val_audio_loader,
        epochs=AUDIO_FULL_EPOCHS,
        learning_rate=AUDIO_LEARNING_RATE,
    )
    net6_overnight_test_summary = test_audio_model(net6_overnight_model, test_audio_loader)
    append_audio_result("Net6LSTM_GANAug", AUDIO_FULL_EPOCHS, "overnight", net6_overnight_train_summary, net6_overnight_test_summary)

    # 5) Save results
    audio_results_df = pd.DataFrame(audio_results_records)
    audio_results_df.to_csv("results_audio_models.csv", index=False)

    # 6) Final summary table
    print("Overnight run complete. Final audio results:")
    display(audio_results_df)
